Parte I de Correos Masivos CC "CARTAS L1"

In [ ]:
# -*- coding: utf-8 -*-
#Se importan las librerias para usarlas.
import smtplib
import email.message
from string import Template
import pyodbc
import pandas as pd
from collections import Counter
from datetime import datetime, timedelta, date
import win32com.client as win32
# Crear una instancia de la aplicación de Outlook
outlook2 = win32.Dispatch("Outlook.Application")
#Conexion a BD en SQL SERVER.
msg = email.message.Message()
con_obj=pyodbc.connect("DRIVER={SQL Server};SERVER=SVDESLW167;DATABASE=BDRIESGO;TRUSTED_CONNECTION=yes")
data_obj=con_obj.cursor()
##Query para obtener el input de las cartas y valor maximo de numeracion.
TodayDate4 = datetime.today().strftime('%Y-%m-%d')
query = "SELECT * FROM PFYCC_GI_REPORTE_CARTAS_CC WHERE Tipo_Linea = 'L1' AND Cod_Carta = ''"
query1 = "SELECT MAX(Valor_carta) FROM PFYCC_GI_REPORTE_MAX_VALOR_CARTA"
#Se lee los datos de las querys, y se crea una variable.
dato_query = pd.read_sql(query, con_obj)
dato_query1 = pd.read_sql(query1,con_obj)
dato_sql  = dato_query.head(100000)
dato_sql1= dato_query1.head(10000)
#Se coloca los datos en un cuadro de datos(Dataframe)
sql_dato = pd.DataFrame(dato_sql)
sql_dato1 = pd.DataFrame(dato_sql1)
"""FERIADO - QUERY PARA COLOCAR LA FECHA DE CC EN EL ASUNTO DE LAS CARTAS"""
"""PARA FERIADOS MODIFICAR LAS QUERYS DE ABAJO"""
query2 = "select TOP 1 FechaCC FROM PFYCC_GI_REPORTE_CARTAS_CC GROUP BY FechaCC ORDER BY 1 DESC" #ULTIMA FECHA
query3 = "select TOP 1 FechaCC FROM PFYCC_GI_REPORTE_CARTAS_CC GROUP BY FechaCC ORDER BY 1 ASC" #primera fecha
#Se lee los datos de las querys, y se crea una variable.
Utl_Fecha= pd.read_sql(query2, con_obj)
Pri_Fecha = pd.read_sql(query3,con_obj)
Utlm_Fecha = Utl_Fecha.head(100000)
Prim_Fecha = Pri_Fecha.head(10000)
#Se coloca los datos en un cuadro de datos(Dataframe)
Frame_Utl_Fecha = pd.DataFrame(Utlm_Fecha)
Frame_Pri_Fecha = pd.DataFrame(Prim_Fecha)
#Se obtiene la ultima fecha de CC
Fecha1 = Frame_Utl_Fecha.loc[0,"FechaCC"]
Fecha1 = str(Fecha1)
#Se obtiene la primera fecha de CC
Fecha2 = Frame_Pri_Fecha.loc[0,"FechaCC"]
Fecha2 = str(Fecha2)
#Condicion para colocar en el asunto de las cartas
if Fecha1 == Fecha2:
    Fech = Fecha2
else:
    Fech = Fecha2 + " " + "al" + " " + Fecha1
#Se crea una variable con la fecha de hoy en formato 'AAAA-MM-DD' para colocarlo como Fecha_Envio de las cartas en SQL
Fecha_Envio = datetime.today().strftime('%Y-%m-%d')
#Se obtiene cada RUC de los comercios del dataframe
RUC_com = sql_dato.loc[:,"RUC"]
contador = Counter(RUC_com)
contador_3 = Counter(RUC_com)
repetidos = [elem for elem in contador if contador[elem]>1]
norep1 = [elem for elem in contador_3 if contador_3[elem] == 1]
datt = int(dato_sql1.loc[0])
for x in range(len(repetidos)):
  datt = datt + 1
  carta = "GD" + str(datt)
  cuenta2 = str(datt)
  asu ="Solicitud de Evidencias Contracargos ingresados del"+ " " + Fech + " " + "-" + " "+ "Carta:" + " "+ carta
  ver =  dato_sql["RUC"] == repetidos[x]
  codd_2 = dato_sql.loc[ver == True]
  dff = pd.DataFrame(codd_2)
  dfff = dff.drop(['FechaDev','RazonSocial','NroControl_ROLCase','Tarjeta','MonedaCC','ImporteCC','ARD_ARN','FechaCC','Ecom','Contracargo','Orden','Terminal','Reten_Dev','Entrymode','Contacto_Opera','Contacto_Comercial','Responsable','Tipo_Linea','Correo_Responsable','Flag_correo','RUC','Cod_Carta','Observaciones','Fecha_Envio'],axis=1)
  dfff1 = dfff.rename(columns={'CodComercio':'Codigo','Tarjeta_Encriptada':'Tarjeta','NomComercio':'Nombre Comercial','MonedaTrx':'Moneda','ImporteTrx':'Importe','FechaTrx':'Fecha de Venta','CodAutorizacion':'Codigo de Autorizacion','MotivoCC':'Motivo','ID_Transacción':'ID_Transaccion'})
  cod2 = dff.loc[:,"Correo_Responsable"]
  cod3 = dff.loc[:,"RazonSocial"]
  cod6 = dff.loc[:,"RUC"]
  cod7 = dff.loc[:,"Contacto_Opera"]
  contador_1 = Counter(cod2)
  repetidos_1 = [elem for elem in contador_1 if contador_1[elem]>1]
  contador_4 = Counter(cod3)
  repetidos_2 = [elem for elem in contador_4 if contador_4[elem]>1]
  contador_2 = Counter(cod2)
  norep2 = [elem for elem in contador_2 if contador_2[elem] == 1]
  contador_6 = Counter(cod6)
  repetidos_6 = [elem for elem in contador_6 if contador_6[elem] > 1]
  contador_7 = Counter(cod7)
  repetidos_7 = [elem for elem in contador_7 if contador_7[elem] > 1]
  norep7  = [elem for elem in contador_7 if contador_7[elem] == 1] # nuevo
  correo_to = None
  if len(repetidos_7)==0:
     correo_to = norep7
  else:
     correo_to = repetidos_7
  cccorreo =  "operaciones@izipay.pe"
  msg['From'] = "operaciones@izipay.pe"
  if str(repetidos_6[0]) =='20604068178':
   msg['To'] = str(', '.join(correo_to[0:2]))
   msg['Cc'] = str(', '.join(repetidos_1[0:4]))+","+ str(', '.join(norep2[0:2]))+","+ "operaciones@izipay.pe"
  else:
   msg['To'] = str(', '.join(correo_to[0:2]))
   msg['Cc'] = str(', '.join(repetidos_1[0:4]))+","+ str(', '.join(norep2[0:2]))+","+ "operaciones@izipay.pe"
  msg['Bcc'] = "operaciones@izipay.pe"
  msg['Subject'] = asu
  if str(repetidos_6[0]) =='20462540745' or str(repetidos_6[0]) =='20606050624':
      if len(norep2)>0:
          message = """
<html>
<head>
<meta name="tipo_contenido"  content="text/html;" http-equiv="content-type" charset="utf-8">
   <title>Solicitud de Evidencias Contracargos</title>
   <style type="text/css">
    a {color: #d80a3e;}
  body, #header h1, #header h2, p {margin: 0; padding: 0;}
  #main {border: 1px solid #cfcece;}
  img {display: block;}
  #top-message p, #bottom p {color: #3f4042; font-size: 12px; font-family: Arial, Helvetica, sans-serif; }
  #header h1 {color: #ffffff !important; font-family: "Lucida Grande", sans-serif; font-size: 24px; margin-bottom: 0!important; padding-bottom: 0; }
  #header p {color: #ffffff !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; font-size: 12px;  }
  h5 {margin: 0 0 0.8em 0;}
    h5 {font-size: 18px; color: #444444 !important; font-family: Arial, Helvetica, sans-serif; }
  p {font-size: 12px; color: #444444 !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; line-height: 1.5;}
   </style>
</head>
<body>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;">&nbsp;</span></font></div>
<div align="center" style="text-align:center;">
<table width="1107" border="0" cellspacing="0" cellpadding="0" style="width:664.5pt;">
<tbody><tr>
<td style="padding:0 0 22.5pt 0;">
<div align="center" style="text-align:center;margin:0;"><h1><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EaDJLaR9WkBMsrTyNpMvJb4BEi3ND8EZyG66UhGdrCTkXA?e=ZWXBd4" width="1107" height="144" alt="Visita IZIPAY" id="_x0000_i1026"></span></font></h1></div>
</td>
</tr>
</tbody></table>
</div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Calibri,sans-serif" size="6" color="red" style="font-family: Calibri, sans-serif, serif, EmojiFont;"><span style="font-size:22.5pt;background-color:white;"><b>Hola</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;"><b>$nombre</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Te&nbsp;informamos que hemos recibido reclamos de los Bancos Emisores por las transacciones detalladas a continuaci&oacute;n:</span></font></p></span></font></div>
              <br>
            <table with="2000" border="0" cellspacing="0" cellpadding="0"  align="center">
            <tbody style="text-align:center;margin:0;">
                 <tr height="42" style="text-align:center;margin:0;">
                  $table1
                </tr>
            </tbody>
        </table>
        </br>
<br>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">De acuerdo al procedimiento establecido, deber&aacute;s enviar en un plazo m&aacute;ximo
de 10 d&iacute;as calendarios la documentaci&oacute;n que sustente el consumo/servicio prestado con una breve explicaci&oacute;n del caso, tales como:</span></font></p></span></font></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">1. Orden de pago firmada/autorizada (voucher), en caso de venta presencial</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">2. Comprobante de la venta (Factura o boleta)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">3. Constancia de entrega y detalle del Producto/Servicio</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">4. La autorizaci&oacute;n para proceder con el reembolso al tarjetahabiente, si corresponde.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">5. Comunicaciones sostenidas con el Tarjetahabiente (correos, cartas, redes sociales, etc.) si aplica.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Agradeceremos enviar la documentaci&oacute;n solicitada a $correo1, $correo2 y operaciones@izipay.pe manteniendo el asunto indicado en esta comunicaci&oacute;n. Asimismo, le estaremos informando por este mismo medio el resultado de la gesti&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;background-color:white;">Finalmente, le recordamos que este procedimiento es conforme a lo dispuesto en el Contrato de Afiliaci&oacute;n
del Establecimiento en las Clausulas: Transacciones observadas y/o fraudulentas y Contracargos, el cual podr&aacute; encontrar en nuestra p&aacute;gina web: www.izipay.pe o www.mc.com.pe/contratos.asp (esta &uacute;ltima en caso su afiliaci&oacute;n sea en la modalidad Adquiriente)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Quedamos a tu disposici&oacute;n y para cualquier consulta puedes comunicarte con nuestros canales de atenci&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Saludos cordiales,</span></font></p></div>
</br>
<div align="center" style="text-align:center;">
<table width="750" border="0" cellspacing="0" cellpadding="0" style="width:450pt;max-width:100%;">
<tbody><tr>
<td style="padding:0;">
<div align="center" style="text-align:center;margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EVlwcg7PmfFEtQZLytr7skEBJGjHY130bWV5LGHfGEUk7A?e=vlJrpX" width="1107" height="312" border="0" alt="Visita IZIPAY" id="_x0000_i1025"></span></font></div>
</td>
</tr>
</tbody></table>
</div>
</body>
</html>
"""
          ss =  Template(message).safe_substitute(nombre=str(repetidos_2[0]),correo1=str(', '.join(repetidos_1[0:4])),correo2=str(norep2[0]),table1=dfff1.to_html(index=False).encode('utf-8').decode('utf-8'))
      else:
       if len(norep2)==0:
           message = """
<html>
<head>
<meta name="tipo_contenido"  content="text/html;" http-equiv="content-type" charset="utf-8">
   <title>Solicitud de Evidencias Contracargos</title>
   <style type="text/css">
    a {color: #d80a3e;}
  body, #header h1, #header h2, p {margin: 0; padding: 0;}
  #main {border: 1px solid #cfcece;}
  img {display: block;}
  #top-message p, #bottom p {color: #3f4042; font-size: 12px; font-family: Arial, Helvetica, sans-serif; }
  #header h1 {color: #ffffff !important; font-family: "Lucida Grande", sans-serif; font-size: 24px; margin-bottom: 0!important; padding-bottom: 0; }
  #header p {color: #ffffff !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; font-size: 12px;  }
  h5 {margin: 0 0 0.8em 0;}
    h5 {font-size: 18px; color: #444444 !important; font-family: Arial, Helvetica, sans-serif; }
  p {font-size: 12px; color: #444444 !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; line-height: 1.5;}
   </style>
</head>
<body>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;">&nbsp;</span></font></div>
<div align="center" style="text-align:center;">
<table width="1107" border="0" cellspacing="0" cellpadding="0" style="width:664.5pt;">
<tbody><tr>
<td style="padding:0 0 22.5pt 0;">
<div align="center" style="text-align:center;margin:0;"><h1><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EaDJLaR9WkBMsrTyNpMvJb4BEi3ND8EZyG66UhGdrCTkXA?e=ZWXBd4" width="1107" height="144" alt="Visita IZIPAY" id="_x0000_i1026"></span></font></h1></div>
</td>
</tr>
</tbody></table>
</div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Calibri,sans-serif" size="6" color="red" style="font-family: Calibri, sans-serif, serif, EmojiFont;"><span style="font-size:22.5pt;background-color:white;"><b>Hola</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;"><b>$nombre</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Te&nbsp;informamos que hemos recibido reclamos de los Bancos Emisores por las transacciones detalladas a continuaci&oacute;n:</span></font></p></span></font></div>
              <br>
            <table with="2000" border="0" cellspacing="0" cellpadding="0"  align="center">
            <tbody style="text-align:center;margin:0;">
                 <tr height="42" style="text-align:center;margin:0;">
                  $table1
                </tr>
            </tbody>
        </table>
        </br>
<br>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">De acuerdo al procedimiento establecido, deber&aacute;s enviar en un plazo m&aacute;ximo
de 10 d&iacute;as calendarios la documentaci&oacute;n que sustente el consumo/servicio prestado con una breve explicaci&oacute;n del caso, tales como:</span></font></p></span></font></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">1. Orden de pago firmada/autorizada (voucher), en caso de venta presencial</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">2. Comprobante de la venta (Factura o boleta)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">3. Constancia de entrega y detalle del Producto/Servicio</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">4. La autorizaci&oacute;n para proceder con el reembolso al tarjetahabiente, si corresponde.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">5. Comunicaciones sostenidas con el Tarjetahabiente (correos, cartas, redes sociales, etc.) si aplica.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Agradeceremos enviar la documentaci&oacute;n solicitada a $correo1 y operaciones@izipay.pe manteniendo el asunto indicado en esta comunicaci&oacute;n. Asimismo, le estaremos informando por este mismo medio el resultado de la gesti&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;background-color:white;">Finalmente, le recordamos que este procedimiento es conforme a lo dispuesto en el Contrato de Afiliaci&oacute;n
del Establecimiento en las Clausulas: Transacciones observadas y/o fraudulentas y Contracargos, el cual podr&aacute; encontrar en nuestra p&aacute;gina web: www.izipay.pe o www.mc.com.pe/contratos.asp (esta &uacute;ltima en caso su afiliaci&oacute;n sea en la modalidad Adquiriente)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Quedamos a tu disposici&oacute;n y para cualquier consulta puedes comunicarte con nuestros canales de atenci&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Saludos cordiales,</span></font></p></div>
</br>
<div align="center" style="text-align:center;">
<table width="750" border="0" cellspacing="0" cellpadding="0" style="width:450pt;max-width:100%;">
<tbody><tr>
<td style="padding:0;">
<div align="center" style="text-align:center;margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EVlwcg7PmfFEtQZLytr7skEBJGjHY130bWV5LGHfGEUk7A?e=vlJrpX" width="1107" height="312" border="0" alt="Visita IZIPAY" id="_x0000_i1025"></span></font></div>
</td>
</tr>
</tbody></table>
</div>
</body>
</html>
"""
           ss =  Template(message).safe_substitute(nombre=str(repetidos_2[0]),correo1=str(', '.join(repetidos_1[0:4])),table1=dfff1.to_html(index=False).encode('utf-8').decode('utf-8'))
  else:
    if str(repetidos_6[0]) !='20462540745' or str(repetidos_6[0]) !='20606050624':
      if len(norep2)>0:
          message = """
<html>
<head>
<meta name="tipo_contenido"  content="text/html;" http-equiv="content-type" charset="utf-8">
   <title>Solicitud de Evidencias Contracargos</title>
   <style type="text/css">
    a {color: #d80a3e;}
  body, #header h1, #header h2, p {margin: 0; padding: 0;}
  #main {border: 1px solid #cfcece;}
  img {display: block;}
  #top-message p, #bottom p {color: #3f4042; font-size: 12px; font-family: Arial, Helvetica, sans-serif; }
  #header h1 {color: #ffffff !important; font-family: "Lucida Grande", sans-serif; font-size: 24px; margin-bottom: 0!important; padding-bottom: 0; }
  #header p {color: #ffffff !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; font-size: 12px;  }
  h5 {margin: 0 0 0.8em 0;}
    h5 {font-size: 18px; color: #444444 !important; font-family: Arial, Helvetica, sans-serif; }
  p {font-size: 12px; color: #444444 !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; line-height: 1.5;}
   </style>
</head>
<body>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;">&nbsp;</span></font></div>
<div align="center" style="text-align:center;">
<table width="1107" border="0" cellspacing="0" cellpadding="0" style="width:664.5pt;">
<tbody><tr>
<td style="padding:0 0 22.5pt 0;">
<div align="center" style="text-align:center;margin:0;"><h1><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EaDJLaR9WkBMsrTyNpMvJb4BEi3ND8EZyG66UhGdrCTkXA?e=ZWXBd4" width="1107" height="144" alt="Visita IZIPAY" id="_x0000_i1026"></span></font></h1></div>
</td>
</tr>
</tbody></table>
</div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Calibri,sans-serif" size="6" color="red" style="font-family: Calibri, sans-serif, serif, EmojiFont;"><span style="font-size:22.5pt;background-color:white;"><b>Hola</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;"><b>$nombre</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Te&nbsp;informamos que hemos recibido reclamos de los Bancos Emisores por las transacciones detalladas a continuaci&oacute;n:</span></font></p></span></font></div>
              <br>
            <table with="2000" border="0" cellspacing="0" cellpadding="0"  align="center">
            <tbody style="text-align:center;margin:0;">
                 <tr height="42" style="text-align:center;margin:0;">
                  $table1
                </tr>
            </tbody>
        </table>
        </br>
<br>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">De acuerdo al procedimiento establecido, deber&aacute;s enviar en un plazo m&aacute;ximo
de 5 d&iacute;as habiles la documentaci&oacute;n que sustente el consumo/servicio prestado con una breve explicaci&oacute;n del caso, tales como:</span></font></p></span></font></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">1. Orden de pago firmada/autorizada (voucher), en caso de venta presencial</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">2. Comprobante de la venta (Factura o boleta)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">3. Constancia de entrega y detalle del Producto/Servicio</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">4. La autorizaci&oacute;n para proceder con el reembolso al tarjetahabiente, si corresponde.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">5. Comunicaciones sostenidas con el Tarjetahabiente (correos, cartas, redes sociales, etc.) si aplica.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Agradeceremos enviar la documentaci&oacute;n solicitada a $correo1, $correo2 y operaciones@izipay.pe manteniendo el asunto indicado en esta comunicaci&oacute;n. Asimismo, le estaremos informando por este mismo medio el resultado de la gesti&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;background-color:white;">Finalmente, le recordamos que este procedimiento es conforme a lo dispuesto en el Contrato de Afiliaci&oacute;n
del Establecimiento en las Clausulas: Transacciones observadas y/o fraudulentas y Contracargos, el cual podr&aacute; encontrar en nuestra p&aacute;gina web: www.izipay.pe o www.mc.com.pe/contratos.asp (esta &uacute;ltima en caso su afiliaci&oacute;n sea en la modalidad Adquiriente)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Quedamos a tu disposici&oacute;n y para cualquier consulta puedes comunicarte con nuestros canales de atenci&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Saludos cordiales,</span></font></p></div>
</br>
<div align="center" style="text-align:center;">
<table width="750" border="0" cellspacing="0" cellpadding="0" style="width:450pt;max-width:100%;">
<tbody><tr>
<td style="padding:0;">
<div align="center" style="text-align:center;margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EVlwcg7PmfFEtQZLytr7skEBJGjHY130bWV5LGHfGEUk7A?e=vlJrpX" width="1107" height="312" border="0" alt="Visita IZIPAY" id="_x0000_i1025"></span></font></div>
</td>
</tr>
</tbody></table>
</div>
</body>
</html>
"""
          ss =  Template(message).safe_substitute(nombre=str(repetidos_2[0]),correo1=str(', '.join(repetidos_1[0:4])),correo2=str(norep2[0]),table1=dfff1.to_html(index=False).encode('utf-8').decode('utf-8'))
      else:
        if len(norep2)==0:
           message = """
<html>
<head>
<meta name="tipo_contenido"  content="text/html;" http-equiv="content-type" charset="utf-8">
   <title>Solicitud de Evidencias Contracargos</title>
   <style type="text/css">
    a {color: #d80a3e;}
  body, #header h1, #header h2, p {margin: 0; padding: 0;}
  #main {border: 1px solid #cfcece;}
  img {display: block;}
  #top-message p, #bottom p {color: #3f4042; font-size: 12px; font-family: Arial, Helvetica, sans-serif; }
  #header h1 {color: #ffffff !important; font-family: "Lucida Grande", sans-serif; font-size: 24px; margin-bottom: 0!important; padding-bottom: 0; }
  #header p {color: #ffffff !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; font-size: 12px;  }
  h5 {margin: 0 0 0.8em 0;}
    h5 {font-size: 18px; color: #444444 !important; font-family: Arial, Helvetica, sans-serif; }
  p {font-size: 12px; color: #444444 !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; line-height: 1.5;}
   </style>
</head>
<body>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;">&nbsp;</span></font></div>
<div align="center" style="text-align:center;">
<table width="1107" border="0" cellspacing="0" cellpadding="0" style="width:664.5pt;">
<tbody><tr>
<td style="padding:0 0 22.5pt 0;">
<div align="center" style="text-align:center;margin:0;"><h1><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EaDJLaR9WkBMsrTyNpMvJb4BEi3ND8EZyG66UhGdrCTkXA?e=ZWXBd4" width="1107" height="144" alt="Visita IZIPAY" id="_x0000_i1026"></span></font></h1></div>
</td>
</tr>
</tbody></table>
</div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Calibri,sans-serif" size="6" color="red" style="font-family: Calibri, sans-serif, serif, EmojiFont;"><span style="font-size:22.5pt;background-color:white;"><b>Hola</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;"><b>$nombre</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Te&nbsp;informamos que hemos recibido reclamos de los Bancos Emisores por las transacciones detalladas a continuaci&oacute;n:</span></font></p></span></font></div>
              <br>
            <table with="2000" border="0" cellspacing="0" cellpadding="0"  align="center">
            <tbody style="text-align:center;margin:0;">
                 <tr height="42" style="text-align:center;margin:0;">
                  $table1
                </tr>
            </tbody>
        </table>
        </br>
<br>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">De acuerdo al procedimiento establecido, deber&aacute;s enviar en un plazo m&aacute;ximo
de 5 d&iacute;as habiles la documentaci&oacute;n que sustente el consumo/servicio prestado con una breve explicaci&oacute;n del caso, tales como:</span></font></p></span></font></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">1. Orden de pago firmada/autorizada (voucher), en caso de venta presencial</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">2. Comprobante de la venta (Factura o boleta)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">3. Constancia de entrega y detalle del Producto/Servicio</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">4. La autorizaci&oacute;n para proceder con el reembolso al tarjetahabiente, si corresponde.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">5. Comunicaciones sostenidas con el Tarjetahabiente (correos, cartas, redes sociales, etc.) si aplica.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Agradeceremos enviar la documentaci&oacute;n solicitada a $correo1 y operaciones@izipay.pe manteniendo el asunto indicado en esta comunicaci&oacute;n. Asimismo, le estaremos informando por este mismo medio el resultado de la gesti&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;background-color:white;">Finalmente, le recordamos que este procedimiento es conforme a lo dispuesto en el Contrato de Afiliaci&oacute;n
del Establecimiento en las Clausulas: Transacciones observadas y/o fraudulentas y Contracargos, el cual podr&aacute; encontrar en nuestra p&aacute;gina web: www.izipay.pe o www.mc.com.pe/contratos.asp (esta &uacute;ltima en caso su afiliaci&oacute;n sea en la modalidad Adquiriente)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Quedamos a tu disposici&oacute;n y para cualquier consulta puedes comunicarte con nuestros canales de atenci&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Saludos cordiales,</span></font></p></div>
</br>
<div align="center" style="text-align:center;">
<table width="750" border="0" cellspacing="0" cellpadding="0" style="width:450pt;max-width:100%;">
<tbody><tr>
<td style="padding:0;">
<div align="center" style="text-align:center;margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EVlwcg7PmfFEtQZLytr7skEBJGjHY130bWV5LGHfGEUk7A?e=vlJrpX" width="1107" height="312" border="0" alt="Visita IZIPAY" id="_x0000_i1025"></span></font></div>
</td>
</tr>
</tbody></table>
</div>
</body>
</html>
"""
           ss =  Template(message).safe_substitute(nombre=str(repetidos_2[0]),correo1=str(', '.join(repetidos_1[0:4])),table1=dfff1.to_html(index=False).encode('utf-8').decode('utf-8'))
  ruc = str(repetidos_6[0])
  linea = str("L1")
  message1 = ss
  msg.add_header('content-type', 'text/html')
  msg.set_payload(message1)

  mail = outlook2.CreateItem(0)
  mail.To = msg['To'].replace('\xa0', '').replace(',',';')#.replace('recepcionsalaverry@costadelsol','recepcionsalaverry@costadelsolperu.com')#.replace('QUISPEUSCAMAYTAROSAHORA@GMAIL.','QUISPEUSCAMAYTAROSAHORA@GMAIL.COM').replace('dianah.salas@hotelesestelar.co','dianah.salas@hotelesestelar.com')
  mail.CC = msg['CC'].replace('\xa0', '').replace(',',';') #"operaciones@izipay.pe"
  mail.Subject = asu
  mail.SentOnBehalfOfName = 'operaciones@izipay.pe'
  mail.HTMLBody = message1
  print(mail.To)
  mail.Send()
  print(carta, asu)
  print("successfully sent email to %s" % (msg['To']))

  del  msg['To']
  del  msg['Cc']
  del  msg['Bcc']
  cusorupdate = con_obj.cursor()
  cusorupdate1 = con_obj.cursor()
  consulta = "UPDATE PFYCC_GI_REPORTE_CARTAS_CC SET Cod_Carta = ?, Fecha_Envio = ? WHERE RUC = ? AND Tipo_Linea = ?;"
  consulta_4 = "INSERT INTO PFYCC_GI_REPORTE_MAX_VALOR_CARTA(Valor_carta)  VALUES ("+cuenta2+");"
  cusorupdate.execute(consulta,str(carta),str(TodayDate4),str(ruc),str(linea))
  cusorupdate.commit()
  cusorupdate.close()
  cusorupdate1.execute(consulta_4)
  cusorupdate1.commit()
  cusorupdate1.close()
query_1 = "SELECT MAX(Valor_carta) FROM PFYCC_GI_REPORTE_MAX_VALOR_CARTA"
dato_1 = pd.read_sql(query_1,con_obj)
dato_2 = dato_1.head(10000)
dato_3 = pd.DataFrame(dato_2)
dato_4 = int(dato_3.loc[0])
for y in range(len(norep1)):
    dato_4 = dato_4 + 1
    carta1 = "GD" + str(dato_4)
    cuenta3 = str(dato_4)
    asu1 ="Solicitud de Evidencias Contracargos ingresados del"+ " " + Fech + " " + "-" + " "+ "Carta:" + " "+ carta1
    ver7 =  dato_sql["RUC"] == norep1[y]
    codd_30 = dato_sql.loc[ver7 == True]
    dff1 = pd.DataFrame(codd_30)
    dfff_1 = dff1.drop(['FechaDev','RazonSocial','NroControl_ROLCase','Tarjeta','MonedaCC','ImporteCC','ARD_ARN','FechaCC','Ecom','Contracargo','Orden','Terminal','Reten_Dev','Entrymode','Contacto_Opera','Contacto_Comercial','Responsable','Tipo_Linea','Correo_Responsable','Flag_correo','RUC','Cod_Carta','Observaciones','Fecha_Envio'],axis=1)
    dfff2 = dfff_1.rename(columns={'CodComercio':'Codigo','Tarjeta_Encriptada':'Tarjeta','NomComercio':'Nombre Comercial','MonedaTrx':'Moneda','ImporteTrx':'Importe','FechaTrx':'Fecha de Venta','CodAutorizacion':'Codigo de Autorizacion','MotivoCC':'Motivo','ID_Transacción':'ID_Transaccion'})
    codd1 = dff1.loc[:,"Correo_Responsable"]
    codd2 = dff1.loc[:,"RazonSocial"]
    codd3 = dff1.loc[:,"RUC"]
    codd4 = dff1.loc[:,"Contacto_Opera"]
    contador12 = Counter(codd1)
    norep4 = [elem for elem in contador12 if contador12[elem]==1]
    contador13 = Counter(codd2)
    norep5 = [elem for elem in contador13 if contador13[elem] == 1]
    contador14 = Counter(codd1)
    repetidos_15 = [elem for elem in contador14 if contador14[elem]>1]
    contador14 = Counter(codd3)
    norep6 = [elem for elem in contador14 if contador14[elem] == 1]
    contador15 = Counter(codd4)
    norep7  = [elem for elem in contador15 if contador15[elem] == 1]
    cccorreo =  "operaciones@izipay.pe"
    msg['From'] = "operaciones@izipay.pe"
    msg['To'] = str(', '.join(norep7[0:2]))
    msg['Cc'] = str(', '.join(norep4[0:2]))+","+ "operaciones@izipay.pe"
    msg['Bcc'] = "operaciones@izipay.pe"
    msg['Subject'] = asu1
    if str(norep6[0]) =='20462540745' or str(norep6[0]) =='20606050624':
      message = """
<html>
<head>
<meta name="tipo_contenido"  content="text/html;" http-equiv="content-type" charset="utf-8">
   <title>Solicitud de Evidencias Contracargos</title>
   <style type="text/css">
    a {color: #d80a3e;}
  body, #header h1, #header h2, p {margin: 0; padding: 0;}
  #main {border: 1px solid #cfcece;}
  img {display: block;}
  #top-message p, #bottom p {color: #3f4042; font-size: 12px; font-family: Arial, Helvetica, sans-serif; }
  #header h1 {color: #ffffff !important; font-family: "Lucida Grande", sans-serif; font-size: 24px; margin-bottom: 0!important; padding-bottom: 0; }
  #header p {color: #ffffff !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; font-size: 12px;  }
  h5 {margin: 0 0 0.8em 0;}
    h5 {font-size: 18px; color: #444444 !important; font-family: Arial, Helvetica, sans-serif; }
  p {font-size: 12px; color: #444444 !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; line-height: 1.5;}
   </style>
</head>
<body>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;">&nbsp;</span></font></div>
<div align="center" style="text-align:center;">
<table width="1107" border="0" cellspacing="0" cellpadding="0" style="width:664.5pt;">
<tbody><tr>
<td style="padding:0 0 22.5pt 0;">
<div align="center" style="text-align:center;margin:0;"><h1><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EaDJLaR9WkBMsrTyNpMvJb4BEi3ND8EZyG66UhGdrCTkXA?e=ZWXBd4" width="1107" height="144" alt="Visita IZIPAY" id="_x0000_i1026"></span></font></h1></div>
</td>
</tr>
</tbody></table>
</div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Calibri,sans-serif" size="6" color="red" style="font-family: Calibri, sans-serif, serif, EmojiFont;"><span style="font-size:22.5pt;background-color:white;"><b>Hola</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;"><b>$nombre</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Te&nbsp;informamos que hemos recibido reclamos de los Bancos Emisores por las transacciones detalladas a continuaci&oacute;n:</span></font></p></span></font></div>
              <br>
            <table with="2000" border="0" cellspacing="0" cellpadding="0"  align="center">
            <tbody style="text-align:center;margin:0;">
                 <tr height="42" style="text-align:center;margin:0;">
                  $table1
                </tr>
            </tbody>
        </table>
        </br>
<br>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">De acuerdo al procedimiento establecido, deber&aacute;s enviar en un plazo m&aacute;ximo
de 10 d&iacute;as calendarios la documentaci&oacute;n que sustente el consumo/servicio prestado con una breve explicaci&oacute;n del caso, tales como:</span></font></p></span></font></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">1. Orden de pago firmada/autorizada (voucher), en caso de venta presencial</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">2. Comprobante de la venta (Factura o boleta)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">3. Constancia de entrega y detalle del Producto/Servicio</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">4. La autorizaci&oacute;n para proceder con el reembolso al tarjetahabiente, si corresponde.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">5. Comunicaciones sostenidas con el Tarjetahabiente (correos, cartas, redes sociales, etc.) si aplica.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Agradeceremos enviar la documentaci&oacute;n solicitada a $correo3 y operaciones@izipay.pe manteniendo el asunto indicado en esta comunicaci&oacute;n. Asimismo, le estaremos informando por este mismo medio el resultado de la gesti&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;background-color:white;">Finalmente, le recordamos que este procedimiento es conforme a lo dispuesto en el Contrato de Afiliaci&oacute;n
del Establecimiento en las Clausulas: Transacciones observadas y/o fraudulentas y Contracargos, el cual podr&aacute; encontrar en nuestra p&aacute;gina web: www.izipay.pe o www.mc.com.pe/contratos.asp (esta &uacute;ltima en caso su afiliaci&oacute;n sea en la modalidad Adquiriente)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Quedamos a tu disposici&oacute;n y para cualquier consulta puedes comunicarte con nuestros canales de atenci&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Saludos cordiales,</span></font></p></div>
</br>
<div align="center" style="text-align:center;">
<table width="750" border="0" cellspacing="0" cellpadding="0" style="width:450pt;max-width:100%;">
<tbody><tr>
<td style="padding:0;">
<div align="center" style="text-align:center;margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EVlwcg7PmfFEtQZLytr7skEBJGjHY130bWV5LGHfGEUk7A?e=vlJrpX" width="1107" height="312" border="0" alt="Visita IZIPAY" id="_x0000_i1025"></span></font></div>
</td>
</tr>
</tbody></table>
</div>
</body>
</html>
"""
      ss =  Template(message).safe_substitute(nombre=str(norep5[0]),correo3=str(norep4[0]),table1=dfff2.to_html(index=False).encode('utf-8').decode('utf-8'))
    else:
      if str(norep6[0]) !='20462540745' or str(norep6[0]) !='20606050624':
       message = """
<html>
<head>
<meta name="tipo_contenido"  content="text/html;" http-equiv="content-type" charset="utf-8">
   <title>Solicitud de Evidencias Contracargos</title>
   <style type="text/css">
    a {color: #d80a3e;}
  body, #header h1, #header h2, p {margin: 0; padding: 0;}
  #main {border: 1px solid #cfcece;}
  img {display: block;}
  #top-message p, #bottom p {color: #3f4042; font-size: 12px; font-family: Arial, Helvetica, sans-serif; }
  #header h1 {color: #ffffff !important; font-family: "Lucida Grande", sans-serif; font-size: 24px; margin-bottom: 0!important; padding-bottom: 0; }
  #header p {color: #ffffff !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; font-size: 12px;  }
  h5 {margin: 0 0 0.8em 0;}
    h5 {font-size: 18px; color: #444444 !important; font-family: Arial, Helvetica, sans-serif; }
  p {font-size: 12px; color: #444444 !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; line-height: 1.5;}
   </style>
</head>
<body>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;">&nbsp;</span></font></div>
<div align="center" style="text-align:center;">
<table width="1107" border="0" cellspacing="0" cellpadding="0" style="width:664.5pt;">
<tbody><tr>
<td style="padding:0 0 22.5pt 0;">
<div align="center" style="text-align:center;margin:0;"><h1><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EaDJLaR9WkBMsrTyNpMvJb4BEi3ND8EZyG66UhGdrCTkXA?e=ZWXBd4" width="1107" height="144" alt="Visita IZIPAY" id="_x0000_i1026"></span></font></h1></div>
</td>
</tr>
</tbody></table>
</div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Calibri,sans-serif" size="6" color="red" style="font-family: Calibri, sans-serif, serif, EmojiFont;"><span style="font-size:22.5pt;background-color:white;"><b>Hola</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;"><b>$nombre</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Te&nbsp;informamos que hemos recibido reclamos de los Bancos Emisores por las transacciones detalladas a continuaci&oacute;n:</span></font></p></span></font></div>
              <br>
            <table with="2000" border="0" cellspacing="0" cellpadding="0"  align="center">
            <tbody style="text-align:center;margin:0;">
                 <tr height="42" style="text-align:center;margin:0;">
                  $table1
                </tr>
            </tbody>
        </table>
        </br>
<br>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">De acuerdo al procedimiento establecido, deber&aacute;s enviar en un plazo m&aacute;ximo
de 5 d&iacute;as habiles la documentaci&oacute;n que sustente el consumo/servicio prestado con una breve explicaci&oacute;n del caso, tales como:</span></font></p></span></font></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">1. Orden de pago firmada/autorizada (voucher), en caso de venta presencial</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">2. Comprobante de la venta (Factura o boleta)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">3. Constancia de entrega y detalle del Producto/Servicio</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">4. La autorizaci&oacute;n para proceder con el reembolso al tarjetahabiente, si corresponde.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">5. Comunicaciones sostenidas con el Tarjetahabiente (correos, cartas, redes sociales, etc.) si aplica.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Agradeceremos enviar la documentaci&oacute;n solicitada a $correo3 y operaciones@izipay.pe manteniendo el asunto indicado en esta comunicaci&oacute;n. Asimismo, le estaremos informando por este mismo medio el resultado de la gesti&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;background-color:white;">Finalmente, le recordamos que este procedimiento es conforme a lo dispuesto en el Contrato de Afiliaci&oacute;n
del Establecimiento en las Clausulas: Transacciones observadas y/o fraudulentas y Contracargos, el cual podr&aacute; encontrar en nuestra p&aacute;gina web: www.izipay.pe o www.mc.com.pe/contratos.asp (esta &uacute;ltima en caso su afiliaci&oacute;n sea en la modalidad Adquiriente)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Quedamos a tu disposici&oacute;n y para cualquier consulta puedes comunicarte con nuestros canales de atenci&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Saludos cordiales,</span></font></p></div>
</br>
<div align="center" style="text-align:center;">
<table width="750" border="0" cellspacing="0" cellpadding="0" style="width:450pt;max-width:100%;">
<tbody><tr>
<td style="padding:0;">
<div align="center" style="text-align:center;margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EVlwcg7PmfFEtQZLytr7skEBJGjHY130bWV5LGHfGEUk7A?e=vlJrpX" width="1107" height="312" border="0" alt="Visita IZIPAY" id="_x0000_i1025"></span></font></div>
</td>
</tr>
</tbody></table>
</div>
</body>
</html>
"""
       ss =  Template(message).safe_substitute(nombre=str(norep5[0]),correo3=str(norep4[0]),table1=dfff2.to_html(index=False).encode('utf-8').decode('utf-8'))
    ruc1 = str(norep6[0])
    linea1 = str("L1")
    message1 =ss
    msg.add_header('content-type', 'text/html')
    msg.set_payload(message1)
    
    mail = outlook2.CreateItem(0)
    mail.To = msg['To'].replace('\xa0', '').replace(',',';')#.replace('GNMCONCILIACIONES@EXPERTIATRAV','GNMCONCILIACIONES@EXPERTIATRAVEL.COM')#.replace('recepcionsalaverry@costadelsol','recepcionsalaverry@costadelsolperu.com')#.replace('devolucionespasajero@perurail.','devolucionespasajero@perurail.com')#.replace('recepcionarequipa@costadelsolp','recepcionarequipa@costadelsolperu.com')
    mail.CC = msg['CC'].replace('\xa0', '').replace(',',';') #"operaciones@izipay.pe"
    mail.Subject = asu1
    mail.SentOnBehalfOfName = 'operaciones@izipay.pe'
    mail.HTMLBody = message1
    print(mail.To)
    mail.Send()
    print("successfully sent email to %s" % (msg['To']))
  
    print(carta1, asu1)
    del  msg['To']
    del  msg['Cc']
    del  msg['Bcc']
    cusorupdate3 = con_obj.cursor()
    cusorupdate2 = con_obj.cursor()
    consulta_3 = "UPDATE PFYCC_GI_REPORTE_CARTAS_CC SET Cod_Carta = ?, Fecha_Envio = ? WHERE RUC = ? AND Tipo_Linea=?;"
    consulta_2 = "INSERT INTO PFYCC_GI_REPORTE_MAX_VALOR_CARTA(Valor_carta)  VALUES ("+cuenta3+");"
    cusorupdate3.execute(consulta_3,str(carta1),str(TodayDate4),str(ruc1),str(linea1))
    cusorupdate3.commit()
    cusorupdate3.close()
    cusorupdate2.execute(consulta_2)
    cusorupdate2.commit()
    cusorupdate2.close()

Parte II de Correos Masivos CC "CARTAS L3"

In [ ]:
# -*- coding: utf-8 -*-
import smtplib
import email.message
from string import Template
import pyodbc
import pandas as pd
from collections import Counter
from datetime import datetime, timedelta, date
#Conexion a BD en SQL SERVER.
msg = email.message.Message()
con_obj=pyodbc.connect("DRIVER={SQL Server};SERVER=SVDESLW167;DATABASE=BDRIESGO;TRUSTED_CONNECTION=yes")
data_obj=con_obj.cursor()
##Query para obtener el input de las cartas y valor maximo de numeracion.
TodayDate4 = datetime.today().strftime('%Y-%m-%d')
query = "SELECT * FROM PFYCC_GI_REPORTE_CARTAS_CC WHERE Tipo_Linea = 'L3' AND Cod_Carta = ''"
query1 = "SELECT MAX(Valor_carta) FROM PFYCC_GI_REPORTE_MAX_VALOR_CARTA"
#Se lee los datos de las querys, y se crea una variable.
dato_query = pd.read_sql(query, con_obj)
dato_query1 = pd.read_sql(query1,con_obj)
dato_sql  = dato_query.head(100000)
dato_sql1= dato_query1.head(10000)
#Se coloca los datos en un cuadro de datos(Dataframe)
sql_dato = pd.DataFrame(dato_sql)
sql_dato1 = pd.DataFrame(dato_sql1)
"""FERIADO - QUERY PARA COLOCAR LA FECHA DE CC EN EL ASUNTO DE LAS CARTAS"""
"""PARA FERIADOS MODIFICAR LAS QUERYS DE ABAJO"""
query2 = "select TOP 1 FechaCC FROM PFYCC_GI_REPORTE_CARTAS_CC GROUP BY FechaCC ORDER BY 1 DESC" #ULTIMA FECHA
query3 = "select TOP 1 FechaCC FROM PFYCC_GI_REPORTE_CARTAS_CC GROUP BY FechaCC ORDER BY 1 ASC" #primera fecha
#Se lee los datos de las querys, y se crea una variable.
Utl_Fecha= pd.read_sql(query2, con_obj)
Pri_Fecha = pd.read_sql(query3,con_obj)
Utlm_Fecha = Utl_Fecha.head(100000)
Prim_Fecha = Pri_Fecha.head(10000)
#Se coloca los datos en un cuadro de datos(Dataframe)
Frame_Utl_Fecha = pd.DataFrame(Utlm_Fecha)
Frame_Pri_Fecha = pd.DataFrame(Prim_Fecha)
#Se obtiene la ultima fecha de CC
Fecha1 = Frame_Utl_Fecha.loc[0,"FechaCC"]
Fecha1 = str(Fecha1)
#Se obtiene la primera fecha de CC
Fecha2 = Frame_Pri_Fecha.loc[0,"FechaCC"]
Fecha2 = str(Fecha2)
#Condicion para colocar en el asunto de las cartas
if Fecha1 == Fecha2:
    Fech = Fecha2
else:
    Fech = Fecha2 + " " + "al" + " " + Fecha1
#Se crea una variable con la fecha de hoy en formato 'AAAA-MM-DD' para colocarlo como Fecha_Envio de las cartas en SQL
Fecha_Envio = datetime.today().strftime('%Y-%m-%d')
#Se obtiene cada RUC de los comercios del dataframe
RUC_com = sql_dato.loc[:,"RUC"]
contador = Counter(RUC_com)
contador_3 = Counter(RUC_com)
repetidos = [elem for elem in contador if contador[elem]>1]
norep1 = [elem for elem in contador_3 if contador_3[elem] == 1]
datt_1 = int(dato_sql1.loc[0])
for x in range(len(repetidos)):
  datt_1 = datt_1 + 1
  carta_1 = "GD" + str(datt_1)
  cuenta2_1 = str(datt_1)
  asu_1 ="Solicitud de Evidencias Contracargos ingresados del"+ " " + Fech + " " + "-" + " "+ "Carta:" + " "+ carta_1
  ver =  dato_sql["RUC"] == repetidos[x]
  codd_2 = dato_sql.loc[ver == True]
  dff = pd.DataFrame(codd_2)
  dfff = dff.drop(['FechaDev','RazonSocial','NroControl_ROLCase','Tarjeta','MonedaCC','ImporteCC','ARD_ARN','FechaCC','Ecom','Contracargo','Orden','Terminal','Reten_Dev','Entrymode','Contacto_Opera','Contacto_Comercial','Responsable','Tipo_Linea','Correo_Responsable','Flag_correo','RUC','Cod_Carta','Observaciones','Fecha_Envio'],axis=1)
  dfff1 = dfff.rename(columns={'CodComercio':'Codigo','Tarjeta_Encriptada':'Tarjeta','NomComercio':'Nombre Comercial','MonedaTrx':'Moneda','ImporteTrx':'Importe','FechaTrx':'Fecha de Venta','CodAutorizacion':'Codigo de Autorizacion','MotivoCC':'Motivo','ID_Transacción':'ID_Transaccion'})
  cod2 = dff.loc[:,"Correo_Responsable"]
  cod3 = dff.loc[:,"RazonSocial"]
  cod6 = dff.loc[:,"RUC"]
  cod7 = dff.loc[:,"Contacto_Opera"]
  contador_1 = Counter(cod2)
  repetidos_1 = [elem for elem in contador_1 if contador_1[elem]>1]
  contador_4 = Counter(cod3)
  repetidos_2 = [elem for elem in contador_4 if contador_4[elem]>1]
  contador_2 = Counter(cod2)
  norep2 = [elem for elem in contador_2 if contador_2[elem] == 1]
  contador_6 = Counter(cod6)
  repetidos_6 = [elem for elem in contador_6 if contador_6[elem] > 1]
  contador_7 = Counter(cod7)
  repetidos_7 = [elem for elem in contador_7 if contador_7[elem] > 1]
  norep7  = [elem for elem in contador_7 if contador_7[elem] == 1] # nuevo
  correo_to = None
  if len(repetidos_7)==0:
     correo_to = norep7
  else:
     correo_to = repetidos_7
  cccorreo =  "operaciones@izipay.pe"
  msg['From'] = "operaciones@izipay.pe"
  if str(repetidos_6[0]) =='20604068178':
   msg['To'] = str(', '.join(correo_to[0:2]))
   msg['Cc'] = str(', '.join(repetidos_1[0:4]))+","+ str(', '.join(norep2[0:2]))+","+ "operaciones@izipay.pe"
  else:
   msg['To'] = str(', '.join(correo_to[0:2]))
   msg['Cc'] = str(', '.join(repetidos_1[0:4]))+","+ str(', '.join(norep2[0:2]))+","+ "operaciones@izipay.pe"
  msg['Bcc'] = "operaciones@izipay.pe"
  msg['Subject'] = asu_1
  if str(repetidos_6[0]) =='20462540745' or str(repetidos_6[0]) =='20606050624':
      if len(norep2)>0:
          message = """
<html>
<head>
<meta name="tipo_contenido"  content="text/html;" http-equiv="content-type" charset="utf-8">
   <title>Solicitud de Evidencias Contracargos</title>
   <style type="text/css">
    a {color: #d80a3e;}
  body, #header h1, #header h2, p {margin: 0; padding: 0;}
  #main {border: 1px solid #cfcece;}
  img {display: block;}
  #top-message p, #bottom p {color: #3f4042; font-size: 12px; font-family: Arial, Helvetica, sans-serif; }
  #header h1 {color: #ffffff !important; font-family: "Lucida Grande", sans-serif; font-size: 24px; margin-bottom: 0!important; padding-bottom: 0; }
  #header p {color: #ffffff !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; font-size: 12px;  }
  h5 {margin: 0 0 0.8em 0;}
    h5 {font-size: 18px; color: #444444 !important; font-family: Arial, Helvetica, sans-serif; }
  p {font-size: 12px; color: #444444 !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; line-height: 1.5;}
   </style>
</head>
<body>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;">&nbsp;</span></font></div>
<div align="center" style="text-align:center;">
<table width="1107" border="0" cellspacing="0" cellpadding="0" style="width:664.5pt;">
<tbody><tr>
<td style="padding:0 0 22.5pt 0;">
<div align="center" style="text-align:center;margin:0;"><h1><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EaDJLaR9WkBMsrTyNpMvJb4BEi3ND8EZyG66UhGdrCTkXA?e=ZWXBd4" width="1107" height="144" alt="Visita IZIPAY" id="_x0000_i1026"></span></font></h1></div>
</td>
</tr>
</tbody></table>
</div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Calibri,sans-serif" size="6" color="red" style="font-family: Calibri, sans-serif, serif, EmojiFont;"><span style="font-size:22.5pt;background-color:white;"><b>Hola</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;"><b>$nombre</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Te&nbsp;informamos que hemos recibido reclamos de los Bancos Emisores por las transacciones detalladas a continuaci&oacute;n:</span></font></p></span></font></div>
              <br>
            <table with="2000" border="0" cellspacing="0" cellpadding="0"  align="center">
            <tbody style="text-align:center;margin:0;">
                 <tr height="42" style="text-align:center;margin:0;">
                  $table1
                </tr>
            </tbody>
        </table>
        </br>
<br>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">De acuerdo al procedimiento establecido, deber&aacute;s enviar en un plazo m&aacute;ximo
de 10 d&iacute;as calendarios la documentaci&oacute;n que sustente el consumo/servicio prestado con una breve explicaci&oacute;n del caso, tales como:</span></font></p></span></font></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">1. Orden de pago firmada/autorizada (voucher), en caso de venta presencial</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">2. Comprobante de la venta (Factura o boleta)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">3. Constancia de entrega y detalle del Producto/Servicio</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">4. La autorizaci&oacute;n para proceder con el reembolso al tarjetahabiente, si corresponde.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">5. Comunicaciones sostenidas con el Tarjetahabiente (correos, cartas, redes sociales, etc.) si aplica.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Agradeceremos enviar la documentaci&oacute;n solicitada a $correo1, $correo2 y operaciones@izipay.pe manteniendo el asunto indicado en esta comunicaci&oacute;n. Asimismo, le estaremos informando por este mismo medio el resultado de la gesti&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;background-color:white;">Finalmente, le recordamos que este procedimiento es conforme a lo dispuesto en el Contrato de Afiliaci&oacute;n
del Establecimiento en las Clausulas: Transacciones observadas y/o fraudulentas y Contracargos, el cual podr&aacute; encontrar en nuestra p&aacute;gina web: www.izipay.pe o www.mc.com.pe/contratos.asp (esta &uacute;ltima en caso su afiliaci&oacute;n sea en la modalidad Adquiriente)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Quedamos a tu disposici&oacute;n y para cualquier consulta puedes comunicarte con nuestros canales de atenci&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Saludos cordiales,</span></font></p></div>
</br>
<div align="center" style="text-align:center;">
<table width="750" border="0" cellspacing="0" cellpadding="0" style="width:450pt;max-width:100%;">
<tbody><tr>
<td style="padding:0;">
<div align="center" style="text-align:center;margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EVlwcg7PmfFEtQZLytr7skEBJGjHY130bWV5LGHfGEUk7A?e=vlJrpX" width="1107" height="312" border="0" alt="Visita IZIPAY" id="_x0000_i1025"></span></font></div>
</td>
</tr>
</tbody></table>
</div>
</body>
</html>
"""
          ss =  Template(message).safe_substitute(nombre=str(repetidos_2[0]),correo1=str(', '.join(repetidos_1[0:4])),correo2=str(norep2[0]),table1=dfff1.to_html(index=False).encode('utf-8').decode('utf-8'))
      else:
       if len(norep2)==0:
           message = """
<html>
<head>
<meta name="tipo_contenido"  content="text/html;" http-equiv="content-type" charset="utf-8">
   <title>Solicitud de Evidencias Contracargos</title>
   <style type="text/css">
    a {color: #d80a3e;}
  body, #header h1, #header h2, p {margin: 0; padding: 0;}
  #main {border: 1px solid #cfcece;}
  img {display: block;}
  #top-message p, #bottom p {color: #3f4042; font-size: 12px; font-family: Arial, Helvetica, sans-serif; }
  #header h1 {color: #ffffff !important; font-family: "Lucida Grande", sans-serif; font-size: 24px; margin-bottom: 0!important; padding-bottom: 0; }
  #header p {color: #ffffff !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; font-size: 12px;  }
  h5 {margin: 0 0 0.8em 0;}
    h5 {font-size: 18px; color: #444444 !important; font-family: Arial, Helvetica, sans-serif; }
  p {font-size: 12px; color: #444444 !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; line-height: 1.5;}
   </style>
</head>
<body>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;">&nbsp;</span></font></div>
<div align="center" style="text-align:center;">
<table width="1107" border="0" cellspacing="0" cellpadding="0" style="width:664.5pt;">
<tbody><tr>
<td style="padding:0 0 22.5pt 0;">
<div align="center" style="text-align:center;margin:0;"><h1><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EaDJLaR9WkBMsrTyNpMvJb4BEi3ND8EZyG66UhGdrCTkXA?e=ZWXBd4" width="1107" height="144" alt="Visita IZIPAY" id="_x0000_i1026"></span></font></h1></div>
</td>
</tr>
</tbody></table>
</div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Calibri,sans-serif" size="6" color="red" style="font-family: Calibri, sans-serif, serif, EmojiFont;"><span style="font-size:22.5pt;background-color:white;"><b>Hola</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;"><b>$nombre</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Te&nbsp;informamos que hemos recibido reclamos de los Bancos Emisores por las transacciones detalladas a continuaci&oacute;n:</span></font></p></span></font></div>
              <br>
            <table with="2000" border="0" cellspacing="0" cellpadding="0"  align="center">
            <tbody style="text-align:center;margin:0;">
                 <tr height="42" style="text-align:center;margin:0;">
                  $table1
                </tr>
            </tbody>
        </table>
        </br>
<br>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">De acuerdo al procedimiento establecido, deber&aacute;s enviar en un plazo m&aacute;ximo
de 10 d&iacute;as calendarios la documentaci&oacute;n que sustente el consumo/servicio prestado con una breve explicaci&oacute;n del caso, tales como:</span></font></p></span></font></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">1. Orden de pago firmada/autorizada (voucher), en caso de venta presencial</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">2. Comprobante de la venta (Factura o boleta)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">3. Constancia de entrega y detalle del Producto/Servicio</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">4. La autorizaci&oacute;n para proceder con el reembolso al tarjetahabiente, si corresponde.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">5. Comunicaciones sostenidas con el Tarjetahabiente (correos, cartas, redes sociales, etc.) si aplica.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Agradeceremos enviar la documentaci&oacute;n solicitada a $correo1 y operaciones@izipay.pe manteniendo el asunto indicado en esta comunicaci&oacute;n. Asimismo, le estaremos informando por este mismo medio el resultado de la gesti&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;background-color:white;">Finalmente, le recordamos que este procedimiento es conforme a lo dispuesto en el Contrato de Afiliaci&oacute;n
del Establecimiento en las Clausulas: Transacciones observadas y/o fraudulentas y Contracargos, el cual podr&aacute; encontrar en nuestra p&aacute;gina web: www.izipay.pe o www.mc.com.pe/contratos.asp (esta &uacute;ltima en caso su afiliaci&oacute;n sea en la modalidad Adquiriente)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Quedamos a tu disposici&oacute;n y para cualquier consulta puedes comunicarte con nuestros canales de atenci&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Saludos cordiales,</span></font></p></div>
</br>
<div align="center" style="text-align:center;">
<table width="750" border="0" cellspacing="0" cellpadding="0" style="width:450pt;max-width:100%;">
<tbody><tr>
<td style="padding:0;">
<div align="center" style="text-align:center;margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EVlwcg7PmfFEtQZLytr7skEBJGjHY130bWV5LGHfGEUk7A?e=vlJrpX" width="1107" height="312" border="0" alt="Visita IZIPAY" id="_x0000_i1025"></span></font></div>
</td>
</tr>
</tbody></table>
</div>
</body>
</html>
"""
           ss =  Template(message).safe_substitute(nombre=str(repetidos_2[0]),correo1=str(', '.join(repetidos_1[0:4])),table1=dfff1.to_html(index=False).encode('utf-8').decode('utf-8'))
  else:
    if str(repetidos_6[0]) !='20462540745' or str(repetidos_6[0]) !='20606050624':
      if len(norep2)>0:
          message = """
<html>
<head>
<meta name="tipo_contenido"  content="text/html;" http-equiv="content-type" charset="utf-8">
   <title>Solicitud de Evidencias Contracargos</title>
   <style type="text/css">
    a {color: #d80a3e;}
  body, #header h1, #header h2, p {margin: 0; padding: 0;}
  #main {border: 1px solid #cfcece;}
  img {display: block;}
  #top-message p, #bottom p {color: #3f4042; font-size: 12px; font-family: Arial, Helvetica, sans-serif; }
  #header h1 {color: #ffffff !important; font-family: "Lucida Grande", sans-serif; font-size: 24px; margin-bottom: 0!important; padding-bottom: 0; }
  #header p {color: #ffffff !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; font-size: 12px;  }
  h5 {margin: 0 0 0.8em 0;}
    h5 {font-size: 18px; color: #444444 !important; font-family: Arial, Helvetica, sans-serif; }
  p {font-size: 12px; color: #444444 !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; line-height: 1.5;}
   </style>
</head>
<body>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;">&nbsp;</span></font></div>
<div align="center" style="text-align:center;">
<table width="1107" border="0" cellspacing="0" cellpadding="0" style="width:664.5pt;">
<tbody><tr>
<td style="padding:0 0 22.5pt 0;">
<div align="center" style="text-align:center;margin:0;"><h1><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EaDJLaR9WkBMsrTyNpMvJb4BEi3ND8EZyG66UhGdrCTkXA?e=ZWXBd4" width="1107" height="144" alt="Visita IZIPAY" id="_x0000_i1026"></span></font></h1></div>
</td>
</tr>
</tbody></table>
</div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Calibri,sans-serif" size="6" color="red" style="font-family: Calibri, sans-serif, serif, EmojiFont;"><span style="font-size:22.5pt;background-color:white;"><b>Hola</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;"><b>$nombre</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Te&nbsp;informamos que hemos recibido reclamos de los Bancos Emisores por las transacciones detalladas a continuaci&oacute;n:</span></font></p></span></font></div>
              <br>
            <table with="2000" border="0" cellspacing="0" cellpadding="0"  align="center">
            <tbody style="text-align:center;margin:0;">
                 <tr height="42" style="text-align:center;margin:0;">
                  $table1
                </tr>
            </tbody>
        </table>
        </br>
<br>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">De acuerdo al procedimiento establecido, deber&aacute;s enviar en un plazo m&aacute;ximo
de 5 d&iacute;as habiles la documentaci&oacute;n que sustente el consumo/servicio prestado con una breve explicaci&oacute;n del caso, tales como:</span></font></p></span></font></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">1. Orden de pago firmada/autorizada (voucher), en caso de venta presencial</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">2. Comprobante de la venta (Factura o boleta)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">3. Constancia de entrega y detalle del Producto/Servicio</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">4. La autorizaci&oacute;n para proceder con el reembolso al tarjetahabiente, si corresponde.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">5. Comunicaciones sostenidas con el Tarjetahabiente (correos, cartas, redes sociales, etc.) si aplica.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Agradeceremos enviar la documentaci&oacute;n solicitada a $correo1, $correo2 y operaciones@izipay.pe manteniendo el asunto indicado en esta comunicaci&oacute;n. Asimismo, le estaremos informando por este mismo medio el resultado de la gesti&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;background-color:white;">Finalmente, le recordamos que este procedimiento es conforme a lo dispuesto en el Contrato de Afiliaci&oacute;n
del Establecimiento en las Clausulas: Transacciones observadas y/o fraudulentas y Contracargos, el cual podr&aacute; encontrar en nuestra p&aacute;gina web: www.izipay.pe o www.mc.com.pe/contratos.asp (esta &uacute;ltima en caso su afiliaci&oacute;n sea en la modalidad Adquiriente)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Quedamos a tu disposici&oacute;n y para cualquier consulta puedes comunicarte con nuestros canales de atenci&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Saludos cordiales,</span></font></p></div>
</br>
<div align="center" style="text-align:center;">
<table width="750" border="0" cellspacing="0" cellpadding="0" style="width:450pt;max-width:100%;">
<tbody><tr>
<td style="padding:0;">
<div align="center" style="text-align:center;margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EVlwcg7PmfFEtQZLytr7skEBJGjHY130bWV5LGHfGEUk7A?e=vlJrpX" width="1107" height="312" border="0" alt="Visita IZIPAY" id="_x0000_i1025"></span></font></div>
</td>
</tr>
</tbody></table>
</div>
</body>
</html>
"""
          ss =  Template(message).safe_substitute(nombre=str(repetidos_2[0]),correo1=str(', '.join(repetidos_1[0:4])),correo2=str(norep2[0]),table1=dfff1.to_html(index=False).encode('utf-8').decode('utf-8'))
      else:
        if len(norep2)==0:
           message = """
<html>
<head>
<meta name="tipo_contenido"  content="text/html;" http-equiv="content-type" charset="utf-8">
   <title>Solicitud de Evidencias Contracargos</title>
   <style type="text/css">
    a {color: #d80a3e;}
  body, #header h1, #header h2, p {margin: 0; padding: 0;}
  #main {border: 1px solid #cfcece;}
  img {display: block;}
  #top-message p, #bottom p {color: #3f4042; font-size: 12px; font-family: Arial, Helvetica, sans-serif; }
  #header h1 {color: #ffffff !important; font-family: "Lucida Grande", sans-serif; font-size: 24px; margin-bottom: 0!important; padding-bottom: 0; }
  #header p {color: #ffffff !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; font-size: 12px;  }
  h5 {margin: 0 0 0.8em 0;}
    h5 {font-size: 18px; color: #444444 !important; font-family: Arial, Helvetica, sans-serif; }
  p {font-size: 12px; color: #444444 !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; line-height: 1.5;}
   </style>
</head>
<body>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;">&nbsp;</span></font></div>
<div align="center" style="text-align:center;">
<table width="1107" border="0" cellspacing="0" cellpadding="0" style="width:664.5pt;">
<tbody><tr>
<td style="padding:0 0 22.5pt 0;">
<div align="center" style="text-align:center;margin:0;"><h1><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EaDJLaR9WkBMsrTyNpMvJb4BEi3ND8EZyG66UhGdrCTkXA?e=ZWXBd4" width="1107" height="144" alt="Visita IZIPAY" id="_x0000_i1026"></span></font></h1></div>
</td>
</tr>
</tbody></table>
</div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Calibri,sans-serif" size="6" color="red" style="font-family: Calibri, sans-serif, serif, EmojiFont;"><span style="font-size:22.5pt;background-color:white;"><b>Hola</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;"><b>$nombre</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Te&nbsp;informamos que hemos recibido reclamos de los Bancos Emisores por las transacciones detalladas a continuaci&oacute;n:</span></font></p></span></font></div>
              <br>
            <table with="2000" border="0" cellspacing="0" cellpadding="0"  align="center">
            <tbody style="text-align:center;margin:0;">
                 <tr height="42" style="text-align:center;margin:0;">
                  $table1
                </tr>
            </tbody>
        </table>
        </br>
<br>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">De acuerdo al procedimiento establecido, deber&aacute;s enviar en un plazo m&aacute;ximo
de 5 d&iacute;as habiles la documentaci&oacute;n que sustente el consumo/servicio prestado con una breve explicaci&oacute;n del caso, tales como:</span></font></p></span></font></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">1. Orden de pago firmada/autorizada (voucher), en caso de venta presencial</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">2. Comprobante de la venta (Factura o boleta)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">3. Constancia de entrega y detalle del Producto/Servicio</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">4. La autorizaci&oacute;n para proceder con el reembolso al tarjetahabiente, si corresponde.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">5. Comunicaciones sostenidas con el Tarjetahabiente (correos, cartas, redes sociales, etc.) si aplica.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Agradeceremos enviar la documentaci&oacute;n solicitada a $correo1 y operaciones@izipay.pe manteniendo el asunto indicado en esta comunicaci&oacute;n. Asimismo, le estaremos informando por este mismo medio el resultado de la gesti&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;background-color:white;">Finalmente, le recordamos que este procedimiento es conforme a lo dispuesto en el Contrato de Afiliaci&oacute;n
del Establecimiento en las Clausulas: Transacciones observadas y/o fraudulentas y Contracargos, el cual podr&aacute; encontrar en nuestra p&aacute;gina web: www.izipay.pe o www.mc.com.pe/contratos.asp (esta &uacute;ltima en caso su afiliaci&oacute;n sea en la modalidad Adquiriente)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Quedamos a tu disposici&oacute;n y para cualquier consulta puedes comunicarte con nuestros canales de atenci&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Saludos cordiales,</span></font></p></div>
</br>
<div align="center" style="text-align:center;">
<table width="750" border="0" cellspacing="0" cellpadding="0" style="width:450pt;max-width:100%;">
<tbody><tr>
<td style="padding:0;">
<div align="center" style="text-align:center;margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EVlwcg7PmfFEtQZLytr7skEBJGjHY130bWV5LGHfGEUk7A?e=vlJrpX" width="1107" height="312" border="0" alt="Visita IZIPAY" id="_x0000_i1025"></span></font></div>
</td>
</tr>
</tbody></table>
</div>
</body>
</html>
"""
           ss =  Template(message).safe_substitute(nombre=str(repetidos_2[0]),correo1=str(', '.join(repetidos_1[0:4])),table1=dfff1.to_html(index=False).encode('utf-8').decode('utf-8'))
  ruc3 = str(repetidos_6[0])
  linea3 = str("L3")
  message1 = ss
  msg.add_header('content-type', 'text/html')
  msg.set_payload(message1)
  mail = outlook2.CreateItem(0)
  mail.To = msg['To'].replace('\xa0', '').replace(',',';')
  mail.CC = msg['CC'].replace('\xa0', '').replace(',',';') #"operaciones@izipay.pe"
  mail.Subject = asu_1
  mail.SentOnBehalfOfName = 'operaciones@izipay.pe'
  mail.HTMLBody = message1
  print(mail.To)
  mail.Send()
  print(carta_1, asu_1)
  print ("successfully sent email to %s" % (msg['To']))
  del  msg['To']
  del  msg['Cc']
  del  msg['Bcc']
  cusorupdate = con_obj.cursor()
  cusorupdate1 = con_obj.cursor()
  consulta = "UPDATE PFYCC_GI_REPORTE_CARTAS_CC SET Cod_Carta = ?, Fecha_Envio = ? WHERE RUC = ? AND Tipo_Linea= ?;"
  consulta_4 = "INSERT INTO PFYCC_GI_REPORTE_MAX_VALOR_CARTA(Valor_carta)  VALUES ("+cuenta2_1+");"
  cusorupdate.execute(consulta,str(carta_1),str(TodayDate4),str(ruc3),str(linea3))
  cusorupdate.commit()
  cusorupdate.close()
  cusorupdate1.execute(consulta_4)
  cusorupdate1.commit()
  cusorupdate1.close()
query__1 = "SELECT MAX(Valor_carta) FROM PFYCC_GI_REPORTE_MAX_VALOR_CARTA"
dato__1 = pd.read_sql(query__1,con_obj)
dato__2 = dato__1.head(10000)
dato__3 = pd.DataFrame(dato__2)
dato__4 = int(dato__3.loc[0])
for y in range(len(norep1)):
    dato__4 = dato__4 + 1
    carta1_ = "GD" + str(dato__4)
    cuenta3_ = str(dato__4)
    asu_2 ="Solicitud de Evidencias Contracargos ingresados del"+ " " + Fech + " " + "-" + " "+ "Carta:" + " "+ carta1_
    ver7 =  dato_sql["RUC"] == norep1[y]
    codd_30 = dato_sql.loc[ver7 == True]
    dff1 = pd.DataFrame(codd_30)
    dfff_1 = dff1.drop(['FechaDev','RazonSocial','NroControl_ROLCase','Tarjeta','MonedaCC','ImporteCC','ARD_ARN','FechaCC','Ecom','Contracargo','Orden','Terminal','Reten_Dev','Entrymode','Contacto_Opera','Contacto_Comercial','Responsable','Tipo_Linea','Correo_Responsable','Flag_correo','RUC','Cod_Carta','Observaciones','Fecha_Envio'],axis=1)
    dfff2 = dfff_1.rename(columns={'CodComercio':'Codigo','Tarjeta_Encriptada':'Tarjeta','NomComercio':'Nombre Comercial','MonedaTrx':'Moneda','ImporteTrx':'Importe','FechaTrx':'Fecha de Venta','CodAutorizacion':'Codigo de Autorizacion','MotivoCC':'Motivo','ID_Transacción':'ID_Transaccion'})
    codd1 = dff1.loc[:,"Correo_Responsable"]
    codd2 = dff1.loc[:,"RazonSocial"]
    codd3 = dff1.loc[:,"RUC"]
    codd4 = dff1.loc[:,"Contacto_Opera"]
    contador12 = Counter(codd1)
    norep4 = [elem for elem in contador12 if contador12[elem]==1]
    contador13 = Counter(codd2)
    norep5 = [elem for elem in contador13 if contador13[elem] == 1]
    contador14 = Counter(codd1)
    repetidos_15 = [elem for elem in contador14 if contador14[elem]>1]
    contador14 = Counter(codd3)
    norep6 = [elem for elem in contador14 if contador14[elem] == 1]
    contador15 = Counter(codd4)
    norep7  = [elem for elem in contador15 if contador15[elem] == 1]
    cccorreo =  "operaciones@izipay.pe"
    msg['From'] = "operaciones@izipay.pe"
    msg['To'] = str(', '.join(norep7[0:2]))
    msg['Cc'] = str(', '.join(norep4[0:2]))+","+ "operaciones@izipay.pe"
    msg['Bcc'] = "operaciones@izipay.pe"
    msg['Subject'] = asu_2
    if str(norep6[0]) =='20462540745' or str(norep6[0]) =='20606050624':
      message = """
<html>
<head>
<meta name="tipo_contenido"  content="text/html;" http-equiv="content-type" charset="utf-8">
   <title>Solicitud de Evidencias Contracargos</title>
   <style type="text/css">
    a {color: #d80a3e;}
  body, #header h1, #header h2, p {margin: 0; padding: 0;}
  #main {border: 1px solid #cfcece;}
  img {display: block;}
  #top-message p, #bottom p {color: #3f4042; font-size: 12px; font-family: Arial, Helvetica, sans-serif; }
  #header h1 {color: #ffffff !important; font-family: "Lucida Grande", sans-serif; font-size: 24px; margin-bottom: 0!important; padding-bottom: 0; }
  #header p {color: #ffffff !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; font-size: 12px;  }
  h5 {margin: 0 0 0.8em 0;}
    h5 {font-size: 18px; color: #444444 !important; font-family: Arial, Helvetica, sans-serif; }
  p {font-size: 12px; color: #444444 !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; line-height: 1.5;}
   </style>
</head>
<body>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;">&nbsp;</span></font></div>
<div align="center" style="text-align:center;">
<table width="1107" border="0" cellspacing="0" cellpadding="0" style="width:664.5pt;">
<tbody><tr>
<td style="padding:0 0 22.5pt 0;">
<div align="center" style="text-align:center;margin:0;"><h1><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EaDJLaR9WkBMsrTyNpMvJb4BEi3ND8EZyG66UhGdrCTkXA?e=ZWXBd4" width="1107" height="144" alt="Visita IZIPAY" id="_x0000_i1026"></span></font></h1></div>
</td>
</tr>
</tbody></table>
</div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Calibri,sans-serif" size="6" color="red" style="font-family: Calibri, sans-serif, serif, EmojiFont;"><span style="font-size:22.5pt;background-color:white;"><b>Hola</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;"><b>$nombre</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Te&nbsp;informamos que hemos recibido reclamos de los Bancos Emisores por las transacciones detalladas a continuaci&oacute;n:</span></font></p></span></font></div>
              <br>
            <table with="2000" border="0" cellspacing="0" cellpadding="0"  align="center">
            <tbody style="text-align:center;margin:0;">
                 <tr height="42" style="text-align:center;margin:0;">
                  $table1
                </tr>
            </tbody>
        </table>
        </br>
<br>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">De acuerdo al procedimiento establecido, deber&aacute;s enviar en un plazo m&aacute;ximo
de 10 d&iacute;as calendarios la documentaci&oacute;n que sustente el consumo/servicio prestado con una breve explicaci&oacute;n del caso, tales como:</span></font></p></span></font></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">1. Orden de pago firmada/autorizada (voucher), en caso de venta presencial</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">2. Comprobante de la venta (Factura o boleta)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">3. Constancia de entrega y detalle del Producto/Servicio</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">4. La autorizaci&oacute;n para proceder con el reembolso al tarjetahabiente, si corresponde.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">5. Comunicaciones sostenidas con el Tarjetahabiente (correos, cartas, redes sociales, etc.) si aplica.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Agradeceremos enviar la documentaci&oacute;n solicitada a $correo3 y operaciones@izipay.pe manteniendo el asunto indicado en esta comunicaci&oacute;n. Asimismo, le estaremos informando por este mismo medio el resultado de la gesti&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;background-color:white;">Finalmente, le recordamos que este procedimiento es conforme a lo dispuesto en el Contrato de Afiliaci&oacute;n
del Establecimiento en las Clausulas: Transacciones observadas y/o fraudulentas y Contracargos, el cual podr&aacute; encontrar en nuestra p&aacute;gina web: www.izipay.pe o www.mc.com.pe/contratos.asp (esta &uacute;ltima en caso su afiliaci&oacute;n sea en la modalidad Adquiriente)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Quedamos a tu disposici&oacute;n y para cualquier consulta puedes comunicarte con nuestros canales de atenci&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Saludos cordiales,</span></font></p></div>
</br>
<div align="center" style="text-align:center;">
<table width="750" border="0" cellspacing="0" cellpadding="0" style="width:450pt;max-width:100%;">
<tbody><tr>
<td style="padding:0;">
<div align="center" style="text-align:center;margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EVlwcg7PmfFEtQZLytr7skEBJGjHY130bWV5LGHfGEUk7A?e=vlJrpX" width="1107" height="312" border="0" alt="Visita IZIPAY" id="_x0000_i1025"></span></font></div>
</td>
</tr>
</tbody></table>
</div>
</body>
</html>
"""
      ss =  Template(message).safe_substitute(nombre=str(norep5[0]),correo3=str(norep4[0]),table1=dfff2.to_html(index=False).encode('utf-8').decode('utf-8'))
    else:
      if str(norep6[0]) !='20462540745' or str(norep6[0]) !='20606050624':
       message = """
<html>
<head>
<meta name="tipo_contenido"  content="text/html;" http-equiv="content-type" charset="utf-8">
   <title>Solicitud de Evidencias Contracargos</title>
   <style type="text/css">
    a {color: #d80a3e;}
  body, #header h1, #header h2, p {margin: 0; padding: 0;}
  #main {border: 1px solid #cfcece;}
  img {display: block;}
  #top-message p, #bottom p {color: #3f4042; font-size: 12px; font-family: Arial, Helvetica, sans-serif; }
  #header h1 {color: #ffffff !important; font-family: "Lucida Grande", sans-serif; font-size: 24px; margin-bottom: 0!important; padding-bottom: 0; }
  #header p {color: #ffffff !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; font-size: 12px;  }
  h5 {margin: 0 0 0.8em 0;}
    h5 {font-size: 18px; color: #444444 !important; font-family: Arial, Helvetica, sans-serif; }
  p {font-size: 12px; color: #444444 !important; font-family: "Lucida Grande", "Lucida Sans", "Lucida Sans Unicode", sans-serif; line-height: 1.5;}
   </style>
</head>
<body>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;">&nbsp;</span></font></div>
<div align="center" style="text-align:center;">
<table width="1107" border="0" cellspacing="0" cellpadding="0" style="width:664.5pt;">
<tbody><tr>
<td style="padding:0 0 22.5pt 0;">
<div align="center" style="text-align:center;margin:0;"><h1><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EaDJLaR9WkBMsrTyNpMvJb4BEi3ND8EZyG66UhGdrCTkXA?e=ZWXBd4" width="1107" height="144" alt="Visita IZIPAY" id="_x0000_i1026"></span></font></h1></div>
</td>
</tr>
</tbody></table>
</div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Calibri,sans-serif" size="6" color="red" style="font-family: Calibri, sans-serif, serif, EmojiFont;"><span style="font-size:22.5pt;background-color:white;"><b>Hola</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><h5><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;"><b>$nombre</b></span></font></h5></span></font></div>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Te&nbsp;informamos que hemos recibido reclamos de los Bancos Emisores por las transacciones detalladas a continuaci&oacute;n:</span></font></p></span></font></div>
              <br>
            <table with="2000" border="0" cellspacing="0" cellpadding="0"  align="center">
            <tbody style="text-align:center;margin:0;">
                 <tr height="42" style="text-align:center;margin:0;">
                  $table1
                </tr>
            </tbody>
        </table>
        </br>
<br>
<div style="margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">De acuerdo al procedimiento establecido, deber&aacute;s enviar en un plazo m&aacute;ximo
de 5 d&iacute;as habiles la documentaci&oacute;n que sustente el consumo/servicio prestado con una breve explicaci&oacute;n del caso, tales como:</span></font></p></span></font></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">1. Orden de pago firmada/autorizada (voucher), en caso de venta presencial</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">2. Comprobante de la venta (Factura o boleta)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">3. Constancia de entrega y detalle del Producto/Servicio</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">4. La autorizaci&oacute;n para proceder con el reembolso al tarjetahabiente, si corresponde.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">5. Comunicaciones sostenidas con el Tarjetahabiente (correos, cartas, redes sociales, etc.) si aplica.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Agradeceremos enviar la documentaci&oacute;n solicitada a $correo3 y operaciones@izipay.pe manteniendo el asunto indicado en esta comunicaci&oacute;n. Asimismo, le estaremos informando por este mismo medio el resultado de la gesti&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;background-color:white;">Finalmente, le recordamos que este procedimiento es conforme a lo dispuesto en el Contrato de Afiliaci&oacute;n
del Establecimiento en las Clausulas: Transacciones observadas y/o fraudulentas y Contracargos, el cual podr&aacute; encontrar en nuestra p&aacute;gina web: www.izipay.pe o www.mc.com.pe/contratos.asp (esta &uacute;ltima en caso su afiliaci&oacute;n sea en la modalidad Adquiriente)</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Quedamos a tu disposici&oacute;n y para cualquier consulta puedes comunicarte con nuestros canales de atenci&oacute;n.</span></font></p></div>
<div style="margin-top:14pt;margin-bottom:14pt;"><p><font face="Tahoma,sans-serif" size="4" color="#666666" style="font-family: Tahoma, sans-serif, serif, EmojiFont;"><span style="font-size:13.5pt;">Saludos cordiales,</span></font></p></div>
</br>
<div align="center" style="text-align:center;">
<table width="750" border="0" cellspacing="0" cellpadding="0" style="width:450pt;max-width:100%;">
<tbody><tr>
<td style="padding:0;">
<div align="center" style="text-align:center;margin:0;"><font face="Times New Roman,serif" size="3" style="font-family: &quot;Times New Roman&quot;, serif, serif, EmojiFont;"><span style="font-size:12pt;"><img src="https://pmpmc.sharepoint.com/:i:/s/RepositorioCRM/EVlwcg7PmfFEtQZLytr7skEBJGjHY130bWV5LGHfGEUk7A?e=vlJrpX" width="1107" height="312" border="0" alt="Visita IZIPAY" id="_x0000_i1025"></span></font></div>
</td>
</tr>
</tbody></table>
</div>
</body>
</html>
"""
       ss =  Template(message).safe_substitute(nombre=str(norep5[0]),correo3=str(norep4[0]),table1=dfff2.to_html(index=False).encode('utf-8').decode('utf-8'))
    ruc4 = str(norep6[0])
    linea4 = str("L3")
    message1 =ss
    msg.add_header('content-type', 'text/html')
    msg.set_payload(message1)

    mail = outlook2.CreateItem(0)
    mail.To = msg['To'].replace('\xa0', '').replace(',',';')
    mail.CC = msg['CC'].replace('\xa0', '').replace(',',';') #"operaciones@izipay.pe"
    mail.Subject = asu_2
    mail.SentOnBehalfOfName = 'operaciones@izipay.pe'
    mail.HTMLBody = message1
    print(mail.To)
    mail.Send()

    print(carta1_, asu_2)
    print ("successfully sent email to %s" % (msg['To']))
    del  msg['To']
    del  msg['Cc']
    del  msg['Bcc']
    cusorupdate3 = con_obj.cursor()
    cusorupdate2 = con_obj.cursor()
    consulta_3 = "UPDATE PFYCC_GI_REPORTE_CARTAS_CC SET Cod_Carta = ?, Fecha_Envio = ? WHERE RUC = ? AND Tipo_Linea=?;"
    consulta_2 = "INSERT INTO PFYCC_GI_REPORTE_MAX_VALOR_CARTA(Valor_carta)  VALUES ("+cuenta3_+");"
    cusorupdate3.execute(consulta_3,str(carta1_),str(TodayDate4),str(ruc4),str(linea4))
    cusorupdate3.commit()
    cusorupdate3.close()
    cusorupdate2.execute(consulta_2)
    cusorupdate2.commit()
    cusorupdate2.close()

CARTAS POR COMERCIO - RUCs INTERCORP

In [ ]:
import pyodbc
import warnings
import pandas as pd
import win32com.client as win32
from datetime import datetime


warnings.filterwarnings("ignore")
# Conexión a tu base de datos (ajusta parámetros)
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=SVDESLW167;"
    "DATABASE=BDRIESGO;"
    "TRUSTED_CONNECTION=yes"
  )
# cursor = conn.cursor()

# Query
query = """
SELECT *
FROM dbo.PFYCC_GI_REPORTE_CARTAS_CC_X_COMERCIO
WHERE cod_carta = ''
"""

#cursor.execute(query)
df_cartas_notificar = pd.read_sql(query, conn,dtype=str)
rows = df_cartas_notificar.shape[0]
# Validar resultados
if rows==0:
    print("No hay nada que enviar")
else:
    print(f"Cartas encontradas: {rows}")
    con_obj3 = pyodbc.connect("DRIVER={SQL Server};SERVER=SVDESLW167;DATABASE=BDRIESGO;TRUSTED_CONNECTION=yes")
    data_obj3 = con_obj3.cursor()
    query3 = "select max(Valor_carta) from PFYCC_GI_REPORTE_MAX_VALOR_CARTA"
    df_cod_carta = pd.read_sql(query3,con_obj3,dtype=int)
    max_valor_carta = df_cod_carta.values[0,0]
    print(f"Max valor: {max_valor_carta}")
    fecha_asunto = df_cartas_notificar['FechaCC'].min() + " al " + df_cartas_notificar['FechaCC'].max()

    #### ENVIO NOTIFICACIÓN POR CORREO
    # --- CONFIGURACIONES ---
    OFT_TEMPLATE_PATH = r"C:\PlantillasOutlook\Plantilla_1era_notif_L1.oft"  # ← Guardar el correo base como .oft
    OUTLOOK_SEND = True  # True para enviar, False para solo mostrar

    # Conexión a la base donde está la secuencia (Windows Auth)
    conn_seq = pyodbc.connect(
        "DRIVER={SQL Server};SERVER=SVDESLW167;DATABASE=BDRIESGO;Trusted_Connection=yes"
    )
    cur_seq = conn_seq.cursor()

    def actualizar_secuencia(cursor, nueva_secuencia):
        cursor.execute("""
            INSERT INTO dbo.PFYCC_GI_REPORTE_MAX_VALOR_CARTA (Valor_carta)
            VALUES (?)
        """, (nueva_secuencia,))
        cursor.connection.commit()

    # Inicializar Outlook
    outlook = win32.Dispatch("Outlook.Application")

    # Ordenar el procesamiento: primero L1, luego L3
    for tipo_linea in ["L1", "L3"]:
        print(f"\nProcesando cartas de Tipo_Linea = {tipo_linea}")

        # Filtrar solo las cartas de ese tipo de línea
        grupos_filtrados = (
            df_cartas_notificar[df_cartas_notificar["Tipo_Linea"] == tipo_linea]
            .groupby(["CodComercio", "Contacto_Opera"], as_index=False)
        )

        for (comercio, destinatario), grupo in grupos_filtrados:
            razsoc = grupo["RazonSocial"].iloc[0] if "RazonSocial" in grupo.columns else ""

            # --- Obtener responsables (correos) sin duplicados ---
            responsables = grupo["Correo_Responsable"].dropna().unique().tolist()
            responsables_cc = ";".join(responsables) if responsables else ""
            responsables_cc_2 = ", ".join(responsables) if responsables else ""

            if isinstance(destinatario, str) and "@" in destinatario:
                # --- Construir tabla HTML excluyendo columnas innecesarias ---
                columnas_para_tabla = [
                    c for c in grupo.columns 
                    if c not in (
                        "ruc","contacto_opera","razsoc","Responsable",
                        "FechaDev","RazonSocial","NroControl_ROLCase",
                        "Tarjeta","MonedaCC","ImporteCC","ARD_ARN",
                        "FechaCC","Ecom","Contracargo","Orden",
                        "Terminal","Reten_Dev","Entrymode","Contacto_Opera",
                        "Contacto_Comercial","Responsable","Tipo_Linea","Correo_Responsable",
                        "Flag_correo","RUC","Cod_Carta","Observaciones","Fecha_Envio"
                    )
                ]
                tabla_html = grupo[columnas_para_tabla].to_html(index=False, border=1)

                # Actualizar secuencia
                max_valor_carta += 1
                secuencia_str = str(max_valor_carta).zfill(3)
                asunto = f"Solicitud de Evidencias Contracargos ingresados del {fecha_asunto} - Carta: GD{secuencia_str}"

                # Crear correo desde plantilla
                mail = outlook.CreateItemFromTemplate(OFT_TEMPLATE_PATH)
                mail.HTMLBody = (
                    mail.HTMLBody
                    .replace("{RazonSocial}", razsoc) # .replace("{<span class=SpellE>Raz_soc</span>}", razsoc)
                    .replace("{Table}", tabla_html)
                    .replace("{responsable}",responsables_cc_2)
                )

                # Asunto y destinatarios
                mail.Subject = asunto
                mail.SentOnBehalfOfName = "operaciones@izipay.pe"
                mail.To = destinatario
                mail.CC = "operaciones@izipay.pe" + (f";{responsables_cc}" if responsables_cc else "")

                # Enviar o mostrar
                if OUTLOOK_SEND:
                    mail.Send()
                    print(f"Enviado carta GD{max_valor_carta} correctamente a {razsoc}, al correo: {destinatario}")
                else:
                    mail.Display()
                    print(f"NO Enviado carta GD{max_valor_carta} a {razsoc}, al correo: {destinatario}")
                # Actualiza tabla cartas
                try:
                    consulta = """
                        UPDATE PFYCC_GI_REPORTE_CARTAS_CC_X_COMERCIO
                        SET Cod_Carta = ?, Fecha_Envio = ?
                        WHERE CodComercio = ? AND Tipo_Linea = ?
                    """
                    fecha_envio = datetime.now().strftime("%Y-%m-%d")
                    cur_seq.execute(
                        consulta,
                        (f"GD{str(max_valor_carta).zfill(3)}", fecha_envio, comercio, tipo_linea)
                    )
                    cur_seq.connection.commit()
                    print(f"Actualizada tabla de cartas para comercio {comercio}, Tipo_Linea {tipo_linea}")
                except Exception as e:
                    print(f"Error actualizando PFYCC_GI_REPORTE_CARTAS_CC_X_COMERCIO: {e}")
                    continue
                # Actualizar secuencia en BD
                try:
                    actualizar_secuencia(cur_seq, str(max_valor_carta))
                except Exception as e:
                    print(f"Error actualizando secuencia después de enviar a {destinatario}: {e}")
                    continue
            else:
                print(f"Destinatario NULL o no válido. Revisar comercio: {comercio}.")
# Cerrar conexión
conn.close()